In [2]:
pip install SensaLisa


Note: you may need to restart the kernel to use updated packages.


# Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from SensaLISA import LISASensitivity


# Generate sensitivity

In [2]:
# Create a standalone LISA sensitivity curve.
lisa = LISASensitivity(
    observation_time="4yr",
    minimum_frequency=1e-5,
    maximum_frequency=1.0,
    number_of_frequencies=1000,
)

In [3]:
# Retrieve different noise quantities.
frequencies, psd = lisa.get_psd()
_, asd = lisa.get_asd()
_, characteristic_strain = lisa.get_characteristic_strain()

print("Number of frequencies:", len(frequencies))
print(f"Minimum frequency: {frequencies[0]:.4e} Hz")
print(f"Maximum frequency: {frequencies[-1]:.4e} Hz")

print("\nFirst five frequencies:")
print(frequencies[:5])

print("\nFirst five PSD values:")
print(psd[:5])

print("\nFirst five ASD values:")
print(asd[:5])

Number of frequencies: 1000
Minimum frequency: 1.0000e-05 Hz
Maximum frequency: 1.0000e+00 Hz

First five frequencies:
[1.00000000e-05 1.01159111e-05 1.02331658e-05 1.03517796e-05
 1.04717682e-05]

First five PSD values:
[1.97250073e-27 1.84074882e-27 1.71779788e-27 1.60305998e-27
 1.49598646e-27]

First five ASD values:
[4.44128442e-14 4.29039488e-14 4.14463253e-14 4.00382314e-14
 3.86779842e-14]


# Quick validation

In [6]:
assert frequencies.shape == psd.shape
assert frequencies.shape == asd.shape
assert frequencies.shape == characteristic_strain.shape

assert np.all(np.diff(frequencies) > 0)
assert np.all(np.isfinite(psd))
assert np.all(np.isfinite(asd))
assert np.all(np.isfinite(characteristic_strain))

assert np.all(psd > 0)
assert np.all(asd > 0)
assert np.all(characteristic_strain > 0)

assert np.allclose(asd, np.sqrt(psd))
assert np.allclose(
    characteristic_strain,
    np.sqrt(frequencies * psd),
)

print("✓ Standalone sensitivity checks passed.")

✓ Standalone sensitivity checks passed.


# Validation and edge cases

In [7]:
import numpy as np

from SensaLisa import (
    LISASensitivity,
    LISASensitivityFromWaveform,
)

In [8]:
try:
    LISASensitivity(observation_time="10yr")
except ValueError as error:
    print("✓ Invalid observation time correctly rejected:")
    print(error)
else:
    raise AssertionError(
        "Invalid observation time should raise ValueError."
    )

✓ Invalid observation time correctly rejected:
Invalid observation time '10yr'. Choose from: 6mo, 1yr, 2yr, 4yr.


# Invalid frequencies

In [9]:
invalid_frequency_arrays = {
    "zero frequency": np.array([0.0, 1e-3, 1e-2]),
    "negative frequency": np.array([-1e-4, 1e-3]),
    "NaN frequency": np.array([1e-4, np.nan, 1e-2]),
    "infinite frequency": np.array([1e-4, np.inf]),
    "empty array": np.array([]),
}

for name, frequency_array in invalid_frequency_arrays.items():
    try:
        LISASensitivityFromWaveform(
            frequencies=frequency_array,
            observation_time="1yr",
        )
    except ValueError:
        print(f"✓ Correctly rejected: {name}")
    else:
        raise AssertionError(
            f"The package did not reject: {name}"
        )

✓ Correctly rejected: zero frequency
✓ Correctly rejected: negative frequency
✓ Correctly rejected: NaN frequency
✓ Correctly rejected: infinite frequency
✓ Correctly rejected: empty array


# Comparing both classes

In [10]:
standalone = LISASensitivity(
    observation_time="4yr",
    minimum_frequency=1e-5,
    maximum_frequency=1.0,
    number_of_frequencies=1000,
)

frequencies, standalone_psd = standalone.get_psd()

waveform_interface = LISASensitivityFromWaveform(
    frequencies=frequencies,
    observation_time="4yr",
)

returned_frequencies, waveform_psd = (
    waveform_interface.get_psd()
)

assert np.array_equal(
    returned_frequencies,
    frequencies,
)

assert np.allclose(
    waveform_psd,
    standalone_psd,
    rtol=1e-12,
    atol=0.0,
)

print("✓ Both classes produce identical PSD values.")

✓ Both classes produce identical PSD values.


# Regression test with known values

In [13]:
test_frequencies = np.array([
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    1e-1,
])

model = LISASensitivityFromWaveform(
    frequencies=test_frequencies,
    observation_time="4yr",
)

_, calculated_asd = model.get_asd()

print("Frequencies:")
print(test_frequencies)

print("\nCalculated ASD values:")
for frequency, value in zip(test_frequencies, calculated_asd):
    print(f"{frequency:.6e} Hz  ->  {value:.12e} 1/sqrt(Hz)")

Frequencies:
[1.e-05 1.e-04 1.e-03 1.e-02 1.e-01]

Calculated ASD values:
1.000000e-05 Hz  ->  4.441284420265e-14 1/sqrt(Hz)
1.000000e-04 Hz  ->  4.637911047571e-17 1/sqrt(Hz)
1.000000e-03 Hz  ->  3.380623123278e-19 1/sqrt(Hz)
1.000000e-02 Hz  ->  1.201279915418e-20 1/sqrt(Hz)
1.000000e-01 Hz  ->  4.613656490473e-20 1/sqrt(Hz)


In [23]:
reference_asd = np.array([
    4.441284420000e-14,
    4.637911047571e-17,
    3.380623123278e-19,
    1.201279915418e-20,
    4.613656490473e-20,
])

In [24]:
assert calculated_asd.shape == reference_asd.shape

In [25]:
assert np.allclose(
    calculated_asd,
    reference_asd,
    rtol=1e-10,
    atol=0.0,
)

print("✓ Numerical regression test passed.")

✓ Numerical regression test passed.


In [26]:
if reference_asd.size == 0:
    print(
        "Regression test skipped: no independently validated "
        "reference values have been added yet."
    )
else:
    assert calculated_asd.shape == reference_asd.shape

    assert np.allclose(
        calculated_asd,
        reference_asd,
        rtol=1e-10,
        atol=0.0,
    )

    print("✓ Numerical regression test passed.")

✓ Numerical regression test passed.
